# W10 — BI Target Leakage Test & Falsification Experiments

**Purpose**: Address advisor's points 2–3 from the to-do list:

1. **Target Leakage Test** (Section 1): For every BI variable, report correlation with true RUL, cycle index, unit identity, and operating regime. Verify that no BI variable uses future information.

2. **BI Falsification Tests** (Section 2): Run 6 configurations to prove that BI improves prediction because it contains structured information, not because it acts as a proxy or leakage channel:
   - (a) Sensor-only (baseline)
   - (b) Sensor + actual BI
   - (c) Sensor + shuffled BI
   - (d) Sensor + lagged BI
   - (e) Sensor + BI excluding `downtime_penalty`
   - (f) Sensor + `downtime_penalty` only

**Dataset**: FD001 (M1)  
**Author**: Fatima Khadija Benzine  
**Date**: October 2026

In [ ]:
import sys
from pathlib import Path

# Add project paths
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Project modules
from config import RAW_DATA_DIR, RESULTS_DIR
from data_loader import MultiDatasetLoader
from bi_fusion import BIFusionPipeline, BI_FEATURE_NAMES, CATEGORICAL_BI_VARS, CONTINUOUS_BI_VARS
from preprocessing import PreprocessingPipelineBI, create_sliding_windows, evaluate_per_unit
from ml_branch import MLBranch, HybridPredictor
from attention import build_dual_attention_bilstm, extract_attention_weights

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Create results directory for this notebook
RESULTS_W10 = RESULTS_DIR / 'W10_leakage_falsification'
RESULTS_W10.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir:  {RESULTS_W10}")

---
## 0. Load & Fuse Data (FD001)

In [ ]:
# Load C-MAPSS FD001
loader = MultiDatasetLoader()
fd001 = loader.load_fd001()

train_raw = fd001['train'].copy()
test_raw  = fd001['test'].copy()

print(f"\nTrain: {train_raw.shape}, Test: {test_raw.shape}")

In [ ]:
# Fuse with BI data (without encoding categoricals — we want raw values for correlation analysis)
bi_pipeline = BIFusionPipeline()

train_fused = bi_pipeline.fuse(train_raw, 'FD001', split='train', encode_categoricals=False)
test_fused  = bi_pipeline.fuse(test_raw,  'FD001', split='test',  encode_categoricals=False)

print(f"\nFused train: {train_fused.shape}")
print(f"Fused test:  {test_fused.shape}")
print(f"\nBI columns present: {[c for c in BI_FEATURE_NAMES if c in train_fused.columns]}")

---
# SECTION 1: Target Leakage Test

For every BI variable, report:
1. Correlation with **true RUL**
2. Correlation with **cycle index** (time proxy)
3. Correlation with **unit identity** (machine proxy)
4. Correlation with **operating regime** (setting proxy)
5. Whether the variable is computed using any **future information**
6. Whether its update is **available at prediction time**

**Rule**: $b_k(t)$ must depend only on information available at or before $t$.

In [ ]:
# 1.1 — Correlation table: each BI variable vs RUL, cycle, unit

bi_vars = [c for c in BI_FEATURE_NAMES if c in train_fused.columns]

corr_results = []
for var in bi_vars:
    row = {
        'variable': var,
        'corr_RUL':     train_fused[var].corr(train_fused['rul']),
        'corr_cycle':   train_fused[var].corr(train_fused['cycle']),
        'corr_unit_id': train_fused[var].corr(train_fused['unit']),
    }
    # Spearman (rank) correlation with RUL for non-linear relationships
    rho, p_val = stats.spearmanr(train_fused[var].dropna(), 
                                  train_fused.loc[train_fused[var].notna(), 'rul'])
    row['spearman_RUL'] = rho
    row['spearman_p']   = p_val
    
    # Operating condition correlation (settings)
    for s in ['setting_1', 'setting_2', 'setting_3']:
        if s in train_fused.columns:
            row[f'corr_{s}'] = train_fused[var].corr(train_fused[s])
    
    corr_results.append(row)

corr_df = pd.DataFrame(corr_results).set_index('variable')
corr_df = corr_df.round(4)

print("=" * 80)
print("BI VARIABLE CORRELATION TABLE")
print("=" * 80)
display(corr_df)

In [ ]:
# 1.2 — Heatmap visualization

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Pearson correlations
pearson_cols = ['corr_RUL', 'corr_cycle', 'corr_unit_id']
if 'corr_setting_1' in corr_df.columns:
    pearson_cols += ['corr_setting_1', 'corr_setting_2', 'corr_setting_3']

sns.heatmap(corr_df[pearson_cols], annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[0],
            xticklabels=[c.replace('corr_', '') for c in pearson_cols])
axes[0].set_title('Pearson Correlation: BI vars vs Potential Leakage Sources', fontsize=12)
axes[0].set_ylabel('')

# Spearman with RUL (non-linear check)
spearman_data = corr_df[['spearman_RUL']].copy()
spearman_data['significant'] = corr_df['spearman_p'] < 0.05

colors = ['#e74c3c' if abs(v) > 0.3 else '#f39c12' if abs(v) > 0.1 else '#27ae60' 
          for v in spearman_data['spearman_RUL']]

bars = axes[1].barh(range(len(bi_vars)), spearman_data['spearman_RUL'], color=colors)
axes[1].set_yticks(range(len(bi_vars)))
axes[1].set_yticklabels(bi_vars, fontsize=9)
axes[1].set_xlabel('Spearman ρ with RUL')
axes[1].set_title('Spearman Rank Correlation with RUL\n(Red: |ρ|>0.3, Orange: |ρ|>0.1, Green: |ρ|≤0.1)', fontsize=12)
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0.3, color='red', linestyle='--', alpha=0.3)
axes[1].axvline(x=-0.3, color='red', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_W10 / 'fig_leakage_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_leakage_correlations.png")

In [ ]:
# 1.3 — Detailed leakage diagnostic for downtime_penalty (the dominant variable)

print("=" * 80)
print("DETAILED DIAGNOSTIC: downtime_penalty")
print("=" * 80)

dp = train_fused[['unit', 'cycle', 'rul', 'downtime_penalty']].copy()

# Normalized life fraction (0 = start, 1 = failure)
dp['life_fraction'] = dp.groupby('unit')['cycle'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)

# Bin into degradation stages
dp['stage'] = pd.cut(dp['life_fraction'], 
                     bins=[0, 0.33, 0.66, 1.0], 
                     labels=['Healthy', 'Degrading', 'Near-Failure'])

print("\ndowntime_penalty statistics by degradation stage:")
stage_stats = dp.groupby('stage', observed=True)['downtime_penalty'].agg(['mean', 'std', 'min', 'max'])
display(stage_stats)

print(f"\nRatio near-failure/healthy mean: {stage_stats.loc['Near-Failure', 'mean'] / stage_stats.loc['Healthy', 'mean']:.2f}x")

# Within-unit correlation: does downtime_penalty always move with degradation?
unit_corrs = []
for unit_id, grp in dp.groupby('unit'):
    if len(grp) > 5:
        r = grp['downtime_penalty'].corr(grp['rul'])
        unit_corrs.append({'unit': unit_id, 'corr_dp_rul': r})

unit_corr_df = pd.DataFrame(unit_corrs)
print(f"\nWithin-unit correlation (downtime_penalty vs RUL):")
print(f"  Mean:   {unit_corr_df['corr_dp_rul'].mean():.4f}")
print(f"  Median: {unit_corr_df['corr_dp_rul'].median():.4f}")
print(f"  Std:    {unit_corr_df['corr_dp_rul'].std():.4f}")
print(f"  Range:  [{unit_corr_df['corr_dp_rul'].min():.4f}, {unit_corr_df['corr_dp_rul'].max():.4f}]")

In [ ]:
# 1.4 — Temporal precedence check
# Verify: is b_k(t) available at prediction time t?
# i.e., does the BI value at cycle t depend only on info from cycles ≤ t?

print("=" * 80)
print("TEMPORAL PRECEDENCE CHECK")
print("=" * 80)
print("\nFor each BI variable, checking if updates depend on future information.")
print("Method: check if BI value at cycle t correlates with FUTURE RUL changes.")
print()

from bi_fusion import BI_DELTA_CONFIG

precedence_results = []
for var in bi_vars:
    delta = BI_DELTA_CONFIG.get(var, {}).get('delta', 'N/A')
    source = BI_DELTA_CONFIG.get(var, {}).get('source', 'N/A')
    var_type = BI_DELTA_CONFIG.get(var, {}).get('type', 'N/A')
    
    # Check: does b_k(t) predict NEXT period's RUL change better than current?
    # If so, it might use future information
    df_check = train_fused[['unit', 'cycle', var, 'rul']].copy()
    df_check['rul_change'] = df_check.groupby('unit')['rul'].diff(-1)  # future change
    df_check = df_check.dropna()
    
    corr_future = df_check[var].corr(df_check['rul_change']) if len(df_check) > 0 else 0
    
    # Check if value changes are independent of future failure
    df_check['var_change'] = df_check.groupby('unit')[var].diff()
    df_check = df_check.dropna()
    corr_var_change_future = df_check['var_change'].corr(df_check['rul_change']) if len(df_check) > 0 else 0
    
    precedence_results.append({
        'variable': var,
        'source': source,
        'delta': delta,
        'type': var_type,
        'corr_with_future_rul_change': round(corr_future, 4),
        'var_change_corr_future': round(corr_var_change_future, 4),
        'uses_future_info': 'NO',  # By construction in generate_bi_data.py
        'available_at_t': 'YES',   # Forward-filled from last update period
    })

precedence_df = pd.DataFrame(precedence_results).set_index('variable')
display(precedence_df)

print("\n" + "=" * 80)
print("LEAKAGE VERDICT")
print("=" * 80)
print("\nBy design (see generate_bi_data.py):")
print("  - production_priority and downtime_penalty are degradation-CORRELATED")
print("    (they respond to current health state, not future failure)")
print("  - This is INTENTIONAL and REALISTIC: a degraded engine on a critical")
print("    mission incurs higher downtime costs and gets higher priority.")
print("  - Correlation ≠ leakage. Leakage = using information from the future.")
print("  - All BI values at cycle t use only information available at or before t.")
print("\n  → The falsification tests in Section 2 will empirically confirm this.")

In [ ]:
# 1.5 — Save leakage test results

corr_df.to_csv(RESULTS_W10 / 'leakage_correlation_table.csv')
precedence_df.to_csv(RESULTS_W10 / 'leakage_precedence_check.csv')
print("Saved leakage test results to", RESULTS_W10)

---
# SECTION 2: BI Falsification Tests

Six configurations to disentangle BI contribution:

| Config | BI data | What it tests |
|:-------|:--------|:--------------|
| (a) | None | Sensor-only baseline |
| (b) | Actual BI | Full framework performance |
| (c) | Shuffled BI | Is BI providing real signal or just noise/regularization? |
| (d) | Lagged BI | Is the temporal alignment necessary? |
| (e) | BI without downtime_penalty | Does BI help beyond the dominant variable? |
| (f) | downtime_penalty only | How much comes from this single variable? |

**Expected outcomes**:
- If (b) > (a): BI adds value
- If (b) ≈ (c): BI is just noise/regularization → **leakage concern**
- If (b) > (c): BI contains genuine structured information ✓
- If (b) > (d): Temporal alignment matters
- If (e) ≈ (b): Other BI vars contribute even without downtime_penalty
- If (f) ≈ (b): downtime_penalty alone explains the gain → narrow contribution

In [ ]:
# 2.0 — Helper: create modified BI DataFrames

def create_shuffled_bi(bi_df: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    """Shuffle each BI variable independently across all rows (breaks unit-cycle structure)."""
    rng = np.random.RandomState(seed)
    shuffled = bi_df.copy()
    for col in BI_FEATURE_NAMES:
        if col in shuffled.columns:
            shuffled[col] = rng.permutation(shuffled[col].values)
    return shuffled


def create_lagged_bi(bi_df: pd.DataFrame, lag_cycles: int = 50) -> pd.DataFrame:
    """Shift BI values forward by lag_cycles within each unit (simulates delayed updates)."""
    lagged = bi_df.copy()
    for col in BI_FEATURE_NAMES:
        if col in lagged.columns:
            lagged[col] = lagged.groupby('unit_id')[col].shift(lag_cycles)
    # Forward-fill NaN from shifting, then backfill remaining
    bi_cols = [c for c in BI_FEATURE_NAMES if c in lagged.columns]
    lagged[bi_cols] = lagged.groupby('unit_id')[bi_cols].ffill()
    lagged[bi_cols] = lagged.groupby('unit_id')[bi_cols].bfill()
    return lagged


def create_bi_without_downtime(bi_df: pd.DataFrame) -> pd.DataFrame:
    """Remove downtime_penalty from BI data."""
    return bi_df.drop(columns=['downtime_penalty'], errors='ignore')


def create_downtime_only(bi_df: pd.DataFrame) -> pd.DataFrame:
    """Keep only unit_id, cycle, and downtime_penalty."""
    keep = ['unit_id', 'cycle', 'downtime_penalty']
    return bi_df[[c for c in keep if c in bi_df.columns]]


print("Helper functions defined ✓")

In [ ]:
# 2.1 — Prepare all 6 configurations

# Load raw BI data
bi_loader = bi_pipeline.bi_loader
bi_train_raw = bi_loader.load_bi('FD001', split='train')
bi_test_raw  = bi_loader.load_bi('FD001', split='test')

# Create modified BI versions
bi_train_shuffled = create_shuffled_bi(bi_train_raw)
bi_test_shuffled  = create_shuffled_bi(bi_test_raw, seed=123)

bi_train_lagged = create_lagged_bi(bi_train_raw, lag_cycles=50)
bi_test_lagged  = create_lagged_bi(bi_test_raw, lag_cycles=50)

bi_train_no_dp = create_bi_without_downtime(bi_train_raw)
bi_test_no_dp  = create_bi_without_downtime(bi_test_raw)

bi_train_dp_only = create_downtime_only(bi_train_raw)
bi_test_dp_only  = create_downtime_only(bi_test_raw)

print("All BI variants created ✓")
print(f"  Shuffled:   {bi_train_shuffled.shape}")
print(f"  Lagged:     {bi_train_lagged.shape}")
print(f"  No DP:      {bi_train_no_dp.shape}")
print(f"  DP only:    {bi_train_dp_only.shape}")

In [ ]:
# 2.2 — Pipeline runner for each configuration
#
# We use the XGBoost branch only for falsification (faster than full BiLSTM).
# The point is to compare BI configurations, not to beat the best model.

from sklearn.preprocessing import MinMaxScaler

RUL_MAX = 125
WINDOW_SIZE = 30


def run_falsification_config(
    train_sensor: pd.DataFrame,
    test_sensor: pd.DataFrame,
    bi_train: pd.DataFrame = None,
    bi_test: pd.DataFrame = None,
    config_name: str = 'unnamed',
    rul_max: int = RUL_MAX,
    window_size: int = WINDOW_SIZE,
) -> dict:
    """
    Run one falsification experiment.
    
    Returns dict with RMSE, Score, MAE, and per-unit predictions.
    """
    print(f"\n{'='*60}")
    print(f"CONFIG: {config_name}")
    print(f"{'='*60}")
    
    train_df = train_sensor.copy()
    test_df  = test_sensor.copy()
    
    # --- Clip RUL ---
    train_df['rul'] = train_df['rul'].clip(upper=rul_max)
    test_df['rul']  = test_df['rul'].clip(upper=rul_max)
    
    # --- Identify sensor columns ---
    sensor_cols = [c for c in train_df.columns if c.startswith('sensor_')]
    setting_cols = [c for c in train_df.columns if c.startswith('setting_')]
    normalize_cols = sensor_cols + setting_cols
    
    # --- Normalize sensors ---
    scaler = MinMaxScaler()
    train_df[normalize_cols] = scaler.fit_transform(train_df[normalize_cols])
    test_df[normalize_cols]  = scaler.transform(test_df[normalize_cols])
    
    # --- Fuse with BI (if provided) ---
    bi_cols = []
    if bi_train is not None:
        bi_t = bi_train.rename(columns={'unit_id': 'unit'}).copy()
        bi_te = bi_test.rename(columns={'unit_id': 'unit'}).copy()
        
        train_df = train_df.merge(bi_t, on=['unit', 'cycle'], how='left')
        test_df  = test_df.merge(bi_te, on=['unit', 'cycle'], how='left')
        
        # Forward/backward fill
        bi_cols_raw = [c for c in bi_t.columns if c not in ['unit', 'cycle']]
        for col in bi_cols_raw:
            if col in train_df.columns:
                train_df[col] = train_df.groupby('unit')[col].ffill().bfill()
                test_df[col]  = test_df.groupby('unit')[col].ffill().bfill()
        
        # One-hot encode categoricals if present
        cats_present = [c for c in CATEGORICAL_BI_VARS if c in train_df.columns]
        if cats_present:
            for col in cats_present:
                train_df[col] = train_df[col].astype(int)
                test_df[col]  = test_df[col].astype(int)
            train_df = pd.get_dummies(train_df, columns=cats_present)
            test_df  = pd.get_dummies(test_df, columns=cats_present)
            # Align columns
            for col in train_df.columns:
                if col not in test_df.columns:
                    test_df[col] = 0
            test_df = test_df[train_df.columns]
        
        # Normalize continuous BI vars
        cont_bi = [c for c in CONTINUOUS_BI_VARS if c in train_df.columns]
        if cont_bi:
            bi_scaler = MinMaxScaler()
            train_df[cont_bi] = bi_scaler.fit_transform(train_df[cont_bi])
            test_df[cont_bi]  = bi_scaler.transform(test_df[cont_bi])
        
        bi_cols = [c for c in train_df.columns 
                   if c not in sensor_cols + setting_cols + ['unit', 'cycle', 'rul']]
    
    # --- Feature columns ---
    feature_cols = [c for c in train_df.columns if c not in ['unit', 'cycle', 'rul']]
    print(f"  Features: {len(sensor_cols)} sensor + {len(bi_cols)} BI = {len(feature_cols)} total")
    
    # --- Sliding windows ---
    X_train, y_train = create_sliding_windows(
        train_df, window_size=window_size, feature_cols=feature_cols, pad=True
    )
    X_test, y_test = create_sliding_windows(
        test_df, window_size=window_size, feature_cols=feature_cols, pad=True
    )
    
    # --- Train XGBoost ---
    ml = MLBranch(model_type='xgboost', flatten_strategy='statistics', random_state=SEED)
    ml.fit(X_train, y_train, feature_names=feature_cols)
    
    # --- Predict & evaluate ---
    y_pred = ml.predict(X_test)
    results = evaluate_per_unit(
        y_test, y_pred, test_df, window_size=window_size, pad=True
    )
    
    results['config'] = config_name
    results['n_features'] = len(feature_cols)
    results['n_bi_features'] = len(bi_cols)
    results['feature_cols'] = feature_cols
    results['bi_cols'] = bi_cols
    
    print(f"\n  → RMSE={results['rmse_last']:.2f}, Score={results['score_last']:.2f}, MAE={results['mae_last']:.2f}")
    
    return results

print("Pipeline runner defined ✓")

In [ ]:
# 2.3 — Run all 6 configurations

all_results = {}

# (a) Sensor-only baseline
all_results['(a) Sensor only'] = run_falsification_config(
    train_raw, test_raw, bi_train=None, bi_test=None,
    config_name='(a) Sensor only'
)

# (b) Sensor + actual BI
all_results['(b) Sensor + BI'] = run_falsification_config(
    train_raw, test_raw, bi_train=bi_train_raw, bi_test=bi_test_raw,
    config_name='(b) Sensor + actual BI'
)

# (c) Sensor + shuffled BI
all_results['(c) Shuffled BI'] = run_falsification_config(
    train_raw, test_raw, bi_train=bi_train_shuffled, bi_test=bi_test_shuffled,
    config_name='(c) Sensor + shuffled BI'
)

# (d) Sensor + lagged BI
all_results['(d) Lagged BI'] = run_falsification_config(
    train_raw, test_raw, bi_train=bi_train_lagged, bi_test=bi_test_lagged,
    config_name='(d) Sensor + lagged BI (50 cycles)'
)

# (e) Sensor + BI without downtime_penalty
all_results['(e) No downtime_penalty'] = run_falsification_config(
    train_raw, test_raw, bi_train=bi_train_no_dp, bi_test=bi_test_no_dp,
    config_name='(e) Sensor + BI (no downtime_penalty)'
)

# (f) Sensor + downtime_penalty only
all_results['(f) downtime_penalty only'] = run_falsification_config(
    train_raw, test_raw, bi_train=bi_train_dp_only, bi_test=bi_test_dp_only,
    config_name='(f) Sensor + downtime_penalty only'
)

print("\n" + "="*60)
print("ALL 6 CONFIGURATIONS COMPLETE")
print("="*60)

In [ ]:
# 2.4 — Summary table

summary_rows = []
for name, res in all_results.items():
    summary_rows.append({
        'Configuration': name,
        'RMSE': round(res['rmse_last'], 2),
        'Score': round(res['score_last'], 2),
        'MAE': round(res['mae_last'], 2),
        'n_features': res['n_features'],
        'n_BI': res['n_bi_features'],
    })

summary_df = pd.DataFrame(summary_rows)

# Add improvement columns relative to sensor-only
baseline_rmse  = summary_df.loc[0, 'RMSE']
baseline_score = summary_df.loc[0, 'Score']
summary_df['ΔRMSE'] = round(summary_df['RMSE'] - baseline_rmse, 2)
summary_df['ΔScore'] = round(summary_df['Score'] - baseline_score, 2)
summary_df['RMSE_improve_%'] = round((baseline_rmse - summary_df['RMSE']) / baseline_rmse * 100, 1)

print("\n" + "="*80)
print("BI FALSIFICATION — RESULTS SUMMARY (FD001, XGBoost)")
print("="*80)
display(summary_df)

summary_df.to_csv(RESULTS_W10 / 'falsification_summary.csv', index=False)
print(f"\nSaved to {RESULTS_W10 / 'falsification_summary.csv'}")

In [ ]:
# 2.5 — Bar chart comparison

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

configs = summary_df['Configuration'].tolist()
x = range(len(configs))

# Color coding
colors = ['#95a5a6',   # (a) grey — baseline
          '#27ae60',   # (b) green — actual BI
          '#e74c3c',   # (c) red — shuffled
          '#f39c12',   # (d) orange — lagged
          '#3498db',   # (e) blue — no DP
          '#9b59b6']   # (f) purple — DP only

# RMSE
bars1 = axes[0].bar(x, summary_df['RMSE'], color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_ylabel('RMSE (lower is better)', fontsize=11)
axes[0].set_title('RMSE by Configuration', fontsize=13)
axes[0].set_xticks(x)
axes[0].set_xticklabels([c.split(')')[0] + ')' for c in configs], fontsize=10)
# Add value labels
for bar, val in zip(bars1, summary_df['RMSE']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=9)

# Score
bars2 = axes[1].bar(x, summary_df['Score'], color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('NASA Score (lower is better)', fontsize=11)
axes[1].set_title('NASA Score by Configuration', fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels([c.split(')')[0] + ')' for c in configs], fontsize=10)
for bar, val in zip(bars2, summary_df['Score']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val:.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_W10 / 'fig_falsification_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_falsification_comparison.png")

In [ ]:
# 2.6 — Per-unit error comparison: (a) vs (b) vs (c)
# Shows WHERE BI helps and whether shuffled BI degrades uniformly

fig, ax = plt.subplots(figsize=(14, 5))

for name, color, marker in [
    ('(a) Sensor only', '#95a5a6', 'o'),
    ('(b) Sensor + BI', '#27ae60', 's'),
    ('(c) Shuffled BI', '#e74c3c', 'x'),
]:
    res = all_results[name]
    errors = np.abs(res['true_last'] - res['preds_last'])
    unit_ids = res['unit_ids']
    ax.scatter(unit_ids, errors, color=color, marker=marker, 
              alpha=0.7, s=30, label=name)

ax.set_xlabel('Test Unit ID', fontsize=11)
ax.set_ylabel('|Prediction Error|', fontsize=11)
ax.set_title('Per-Unit Absolute Error: Sensor-only vs Actual BI vs Shuffled BI', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_W10 / 'fig_per_unit_error_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2.7 — Interpretation guide

print("="*80)
print("INTERPRETATION GUIDE")
print("="*80)

r = summary_df.set_index('Configuration')
rmse_a = r.loc['(a) Sensor only', 'RMSE']
rmse_b = r.loc['(b) Sensor + BI', 'RMSE']
rmse_c = r.loc['(c) Shuffled BI', 'RMSE']
rmse_d = r.loc['(d) Lagged BI', 'RMSE']
rmse_e = r.loc['(e) No downtime_penalty', 'RMSE']
rmse_f = r.loc['(f) downtime_penalty only', 'RMSE']

print(f"\n1. Does BI add value?")
print(f"   (b) RMSE={rmse_b:.2f} vs (a) RMSE={rmse_a:.2f}")
if rmse_b < rmse_a:
    print(f"   → YES: BI improves RMSE by {rmse_a - rmse_b:.2f} ({(rmse_a-rmse_b)/rmse_a*100:.1f}%)")
else:
    print(f"   → NO improvement from BI")

print(f"\n2. Is BI signal genuine (not noise/regularization)?")
print(f"   (b) RMSE={rmse_b:.2f} vs (c) RMSE={rmse_c:.2f}")
if rmse_b < rmse_c:
    print(f"   → YES: Actual BI outperforms shuffled by {rmse_c - rmse_b:.2f}")
    print(f"     BI contains structured information, not just noise ✓")
else:
    print(f"   → WARNING: Shuffled BI performs similarly → may be regularization artifact")

print(f"\n3. Does temporal alignment matter?")
print(f"   (b) RMSE={rmse_b:.2f} vs (d) RMSE={rmse_d:.2f}")
if rmse_b < rmse_d:
    print(f"   → YES: Properly aligned BI outperforms lagged by {rmse_d - rmse_b:.2f}")
else:
    print(f"   → Alignment does not significantly affect results")

print(f"\n4. Is the contribution broader than downtime_penalty?")
print(f"   (e) without DP: RMSE={rmse_e:.2f} vs (a) sensor-only: RMSE={rmse_a:.2f}")
if rmse_e < rmse_a:
    print(f"   → YES: Other BI vars also contribute (RMSE improves by {rmse_a - rmse_e:.2f})")
else:
    print(f"   → Other BI vars do not improve over sensor-only")

print(f"\n5. How much does downtime_penalty alone explain?")
if rmse_b < rmse_a:
    total_improvement = rmse_a - rmse_b
    dp_contribution = rmse_a - rmse_f
    print(f"   Total BI improvement: {total_improvement:.2f}")
    print(f"   downtime_penalty alone: {dp_contribution:.2f} ({dp_contribution/total_improvement*100:.0f}% of total)")
    remaining = total_improvement - dp_contribution
    print(f"   Other BI variables: {remaining:.2f} ({remaining/total_improvement*100:.0f}% of total)")

In [ ]:
# 2.8 — Statistical significance: paired test between configs

from scipy.stats import wilcoxon

print("="*80)
print("STATISTICAL SIGNIFICANCE (Wilcoxon signed-rank, paired unit-level errors)")
print("="*80)

# Reference: sensor-only errors
ref_errors = np.abs(all_results['(a) Sensor only']['true_last'] - 
                    all_results['(a) Sensor only']['preds_last'])

for name in ['(b) Sensor + BI', '(c) Shuffled BI', '(d) Lagged BI', 
             '(e) No downtime_penalty', '(f) downtime_penalty only']:
    res = all_results[name]
    test_errors = np.abs(res['true_last'] - res['preds_last'])
    
    # Wilcoxon signed-rank test on absolute errors
    try:
        stat, p_value = wilcoxon(ref_errors, test_errors)
        sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
        print(f"  {name:40s} p={p_value:.4f} {sig}")
    except Exception as e:
        print(f"  {name:40s} test failed: {e}")

print("\n  Significance: *** p<0.001, ** p<0.01, * p<0.05, ns: not significant")

# Also test (b) vs (c) directly — the key falsification comparison
print("\n  Direct comparison (b) actual BI vs (c) shuffled BI:")
err_b = np.abs(all_results['(b) Sensor + BI']['true_last'] - all_results['(b) Sensor + BI']['preds_last'])
err_c = np.abs(all_results['(c) Shuffled BI']['true_last'] - all_results['(c) Shuffled BI']['preds_last'])
try:
    stat, p_value = wilcoxon(err_b, err_c)
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
    print(f"  Wilcoxon p={p_value:.4f} {sig}")
except Exception as e:
    print(f"  Test failed: {e}")

---
## Summary

### Leakage Test Results
- `downtime_penalty` and `production_priority` are **degradation-correlated by design** (see `generate_bi_data.py`)
- This is **intentional and realistic**: a degraded engine serving a critical mission incurs higher costs
- **No variable uses future information**: all BI values at cycle $t$ depend only on information available at or before $t$
- Correlation with RUL ≠ leakage; leakage = using future information

### Falsification Test Results
- The results table above demonstrates whether BI provides genuine structured information
- The shuffled BI test is the critical check: if actual BI >> shuffled BI, the contribution is real
- The downtime_penalty isolation tests show the breadth of the BI contribution

### Next Steps
- Replicate with BiLSTM + Dual Attention (not just XGBoost) for the paper's main results
- Consider running with multiple seeds for statistical robustness
- Add results to Section 5 of the paper